# Analyze the MC syst dfs

- Fill NaNs in sphere charge variables and apply cuts on them
- Exposure accounting
- Works for syst knobs: merging (flux and G4), and capping (g4 neutron)
- Make signal region and sideband event dfs
- Optimize binnings
- Define var_config with the optimized bins
- Collect variables for XSEC unit
- Check Signal Histograms
- Produce multiverses for GENIE multisigma and morph knobs
    - Flux and Geant4 knobs are all multiverses
- Flux: weights are merged since there are correlations between the knobs
- Geatn4: should be merged or not?

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime

import uproot
from matplotlib import gridspec

import sys
sys.path.append('../../../../')
from pyanalib.split_df_helpers import *
import pyanalib.pandas_helpers as ph
import pyanalib.stat_helpers as sh
from makedf.util import *
from analysis_village.plot_style.plot_helper import *
from analysis_village.unfolding.covariance import *
from analysis_village.unfolding.utils import *
from analysis_village.unfolding.variable_configs import VariableConfig

from analysis_village.cohpi.unfolding.cohpi_topologies import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

In [ ]:
sample_str = "v10_14_02_02+"

## Open input file

In [ ]:
input_file = "cohpi_aurora_5p4e20_systs.df"

In [ ]:
print_keys(input_file)

In [ ]:
mc_bnb_cosmic_dfs = {}
keys2load = ['evt', 'hdr', 'histgenevtdf', 'histpotdf', 'mcnu', 'pot']
dfs = {}
for key in keys2load:
    print(f"Loading {key}...")
    mc_bnb_cosmic_dfs[key] = pd.read_hdf(input_file, key=f"{key}")
#evt_df, hdr_df, histgenevtdf_df, histpotdf_df, mcnu_df, pot_df = dfs['evt'], dfs['hdr'], dfs['histgenevtdf'], dfs['histpotdf'], dfs['mcnu'], dfs['pot']

In [ ]:
data_bnb_light_dfs = {}
keys2load = ['evt', 'hdr', 'histgenevtdf', 'histpotdf', 'pot']
for key in keys2load:
    this_key = key + "_0"
    print(f"Loading {key}...")

    data_bnb_light_dfs[key] = pd.read_hdf("/data/sungbino/sbnd/gen2/cohpi/syst_data/data_SBND2026A_SBND2026A_gen2_run1_BNBLight_Data_FixedDev_Respin_v10_14_02_04_flatcaf_sbnd_cohpi_all_weight_signal.df", key=f"{this_key}")

In [ ]:
data_offbeam_light_dfs = {}
keys2load = ['evt', 'hdr', 'histgenevtdf', 'histpotdf', 'pot']
for key in keys2load:
    this_key = key + "_0"
    print(f"Loading {key}...")

    data_offbeam_light_dfs[key] = pd.read_hdf("/data/sungbino/sbnd/gen2/cohpi/syst_data/data_SBND2026A_gen2_InTime-Run1_v10_14_02_02_flatcaf_sbnd_cohpi_all_weight_signal.df", key=f"{this_key}")

## fill NaNs

In [ ]:
mc_bnb_cosmic_dfs['evt'][('nuint_categ', '')] = mc_bnb_cosmic_dfs['evt'].nuint_categ.fillna(-2)
mc_bnb_cosmic_dfs['evt'][('sum_integ_1cm', '')] = mc_bnb_cosmic_dfs['evt'].sum_integ_1cm.fillna(0.)
mc_bnb_cosmic_dfs['evt'][('sum_integ_2cm', '')] = mc_bnb_cosmic_dfs['evt'].sum_integ_2cm.fillna(0.)
mc_bnb_cosmic_dfs['evt'][('sum_integ_3cm', '')] = mc_bnb_cosmic_dfs['evt'].sum_integ_3cm.fillna(0.)
mc_bnb_cosmic_dfs['evt'][('sum_integ_4cm', '')] = mc_bnb_cosmic_dfs['evt'].sum_integ_4cm.fillna(0.)
mc_bnb_cosmic_dfs['evt'][('charge_spere_3_over_4m3', '')] = mc_bnb_cosmic_dfs['evt'].sum_integ_3cm / (mc_bnb_cosmic_dfs['evt'].sum_integ_4cm - mc_bnb_cosmic_dfs['evt'].sum_integ_3cm)
mc_bnb_cosmic_dfs['evt'][('charge_spere_3_over_4m3', '')] = mc_bnb_cosmic_dfs['evt'].charge_spere_3_over_4m3.fillna(0.)

In [ ]:
data_offbeam_light_dfs['evt'][('nuint_categ', '')] = -3
data_offbeam_light_dfs['evt'][('sum_integ_1cm', '')] = data_offbeam_light_dfs['evt'].sum_integ_1cm.fillna(0.)
data_offbeam_light_dfs['evt'][('sum_integ_2cm', '')] = data_offbeam_light_dfs['evt'].sum_integ_2cm.fillna(0.)
data_offbeam_light_dfs['evt'][('sum_integ_3cm', '')] = data_offbeam_light_dfs['evt'].sum_integ_3cm.fillna(0.)
data_offbeam_light_dfs['evt'][('sum_integ_4cm', '')] = data_offbeam_light_dfs['evt'].sum_integ_4cm.fillna(0.)
data_offbeam_light_dfs['evt'][('charge_spere_3_over_4m3', '')] = data_offbeam_light_dfs['evt'].sum_integ_3cm / (data_offbeam_light_dfs['evt'].sum_integ_4cm - data_offbeam_light_dfs['evt'].sum_integ_3cm)
data_offbeam_light_dfs['evt'][('charge_spere_3_over_4m3', '')] = data_offbeam_light_dfs['evt'].charge_spere_3_over_4m3.fillna(0.)

In [ ]:
data_bnb_light_dfs['evt'][('sum_integ_1cm', '')] = data_bnb_light_dfs['evt'].sum_integ_1cm.fillna(0.)
data_bnb_light_dfs['evt'][('sum_integ_2cm', '')] = data_bnb_light_dfs['evt'].sum_integ_2cm.fillna(0.)
data_bnb_light_dfs['evt'][('sum_integ_3cm', '')] = data_bnb_light_dfs['evt'].sum_integ_3cm.fillna(0.)
data_bnb_light_dfs['evt'][('sum_integ_4cm', '')] = data_bnb_light_dfs['evt'].sum_integ_4cm.fillna(0.)
data_bnb_light_dfs['evt'][('charge_spere_3_over_4m3', '')] = data_bnb_light_dfs['evt'].sum_integ_3cm / (data_bnb_light_dfs['evt'].sum_integ_4cm - data_bnb_light_dfs['evt'].sum_integ_3cm)
data_bnb_light_dfs['evt'][('charge_spere_3_over_4m3', '')] = data_bnb_light_dfs['evt'].charge_spere_3_over_4m3.fillna(0.)

### Cut on Charges in Spheres

In [ ]:
mc_bnb_cosmic_dfs['evt'] = mc_bnb_cosmic_dfs['evt'][(mc_bnb_cosmic_dfs['evt'].sum_integ_1cm < 4000) & (mc_bnb_cosmic_dfs['evt'].sum_integ_2cm < 7000) & (mc_bnb_cosmic_dfs['evt'].sum_integ_3cm < 10000) & (mc_bnb_cosmic_dfs['evt'].sum_integ_4cm < 13000) & (mc_bnb_cosmic_dfs['evt'].charge_spere_3_over_4m3 < 6.)]
data_offbeam_light_dfs['evt'] = data_offbeam_light_dfs['evt'][(data_offbeam_light_dfs['evt'].sum_integ_1cm < 4000) & (data_offbeam_light_dfs['evt'].sum_integ_2cm < 7000) & (data_offbeam_light_dfs['evt'].sum_integ_3cm < 10000) & (data_offbeam_light_dfs['evt'].sum_integ_4cm < 13000) & (data_offbeam_light_dfs['evt'].charge_spere_3_over_4m3 < 6.)]
data_bnb_light_dfs['evt'] = data_bnb_light_dfs['evt'][(data_bnb_light_dfs['evt'].sum_integ_1cm < 4000) & (data_bnb_light_dfs['evt'].sum_integ_2cm < 7000) & (data_bnb_light_dfs['evt'].sum_integ_3cm < 10000) & (data_bnb_light_dfs['evt'].sum_integ_4cm < 13000) & (data_bnb_light_dfs['evt'].charge_spere_3_over_4m3 < 6.)]

In [ ]:
mc_bnb_cosmic_dfs['evt'].nuint_categ.value_counts()

## Exposure Accounting

In [ ]:
def get_n_evt(df):
    unique_count = df.index.droplevel(
        list(df.index.names[2:])  # drop everything except first two levels
    ).nunique()
    return unique_count

def get_n_evt_mc(df):
    unique_count = df.index.droplevel(
        list(df.index.names[3:])  # drop everything except first two levels
    ).nunique()
    return unique_count

In [ ]:
## Collect the offbeam data fudge factor and scale for offbeam data
n_record_spill_data = get_n_evt(data_bnb_light_dfs['hdr'])
n_gates_data = len(data_bnb_light_dfs["pot"])

n_record_spill_offbeam_data = get_n_evt(data_offbeam_light_dfs['hdr'])
n_gates_offbeam_data = data_offbeam_light_dfs["hdr"][data_offbeam_light_dfs["hdr"]['first_in_subrun'] == 1]['noffbeambnb'].sum()

p_trig_data = n_record_spill_data / n_gates_data
p_trig_offbeam_data = n_record_spill_offbeam_data / n_gates_offbeam_data
print("p_trig_data: %f" %p_trig_data)
print("p_trig_offbeam_data: %f" %p_trig_offbeam_data)

f_factor = (p_trig_data - p_trig_offbeam_data) / (1 - p_trig_offbeam_data)
print("f_factor: %f" %f_factor)

intime_gate_scale = (1. - f_factor) * (n_gates_data + 0.) / (n_gates_offbeam_data + 0.)
print("intime_gate_scale: %f" %intime_gate_scale)


In [ ]:
## Collect pot scale for MC
mc_tot_pot = mc_bnb_cosmic_dfs["hdr"]['pot'].sum()

data_tot_pot = data_bnb_light_dfs["hdr"]['pot'].sum()
data_tot_TOR860 = data_bnb_light_dfs["pot"]['TOR860'].sum()
data_tot_TOR875 = data_bnb_light_dfs["pot"]['TOR875'].sum()

print("mc_tot_pot: %e" %(mc_tot_pot))

print("data_tot_pot: %e" %(data_tot_pot))
print("data_tot_TOR860: %e" %(data_tot_TOR860))
print("data_tot_TOR875: %e" %(data_tot_TOR875))

target_pot = data_tot_pot
mc_pot_scale = target_pot / mc_tot_pot
print("MC POT scale: %.3f" %(mc_pot_scale))

In [ ]:
mc_bnb_cosmic_dfs["hdr"]

In [ ]:
## Comparison between observed and expected total number of recorded spills
n_evt_mc = get_n_evt_mc(mc_bnb_cosmic_dfs["hdr"])

print("n_evt_data_onbeam: %d" %n_record_spill_data)
print("n_evt_exp.: %f" %(n_evt_mc * mc_pot_scale + n_record_spill_offbeam_data * intime_gate_scale))
print("- n_evt_mc: %f" %(n_evt_mc * mc_pot_scale))
print("- n_evt_data_offbeam: %f" %(n_record_spill_offbeam_data * intime_gate_scale))

In [ ]:
mc_bnb_cosmic_dfs['evt']["pot_weight"] = 1. #mc_pot_scale * np.ones(len(mc_bnb_cosmic_dfs['evt']))
mc_bnb_cosmic_dfs['mcnu']["pot_weight"] = 1.#mc_pot_scale * np.ones(len(mc_bnb_cosmic_dfs['mcnu']))

In [ ]:
mc_bnb_cosmic_dfs['mcnu'].pot_weight

In [ ]:
mc_bnb_cosmic_dfs['hdr'].pot.sum()

In [ ]:
mc_bnb_cosmic_dfs['histpotdf'].TotalPOT.sum()

In [ ]:
mc_bnb_cosmic_dfs['mcnu'].nuint_categ.value_counts()

## Works for Syst Knobs

### Collect knob names

In [ ]:
lvl0 = mc_bnb_cosmic_dfs['mcnu'].columns.get_level_values(0)

genie_multisim_knobs  = sorted(set(c for c in lvl0 if c.startswith("GENIEReWeight") and "multisim"  in c))
genie_multisigma_knobs  = sorted(set(c for c in lvl0 if c.startswith("GENIEReWeight") and "multisigma" in c))
flux_multisim_knobs  = sorted(set(c for c in lvl0 if "Flux" in c))
G4_multisim_knobs  = sorted(set(c for c in lvl0 if "Geant4" in c))

print("multisim: %d knobs and they are: %s" % (len(genie_multisim_knobs), genie_multisim_knobs))
print("multisigma: %d knobs and they are: %s" % (len(genie_multisigma_knobs), genie_multisigma_knobs))
print("flux_multisim_knobs: %d knobs and they are: %s" % (len(flux_multisim_knobs), flux_multisim_knobs))
print("G4_multisim_knobs: %d knobs and they are: %s" % (len(G4_multisim_knobs), G4_multisim_knobs))

### GENIE knobs to be used

In [ ]:
genie_multisigma_knobs = [
                            # - QE
                            "GENIEReWeight_SBN_v1_multisigma_VecFFCCQEshape",

                            # - MEC
                            "GENIEReWeight_SBN_v1_multisigma_DecayAngMEC",

                            # - RES
                            "GENIEReWeight_SBN_v1_multisigma_MaCCRES",
                            "GENIEReWeight_SBN_v1_multisigma_MvCCRES",
                            "GENIEReWeight_SBN_v1_multisigma_MaNCRES",
                            "GENIEReWeight_SBN_v1_multisigma_MvNCRES",
                            "GENIEReWeight_SBN_v1_multisigma_Theta_Delta2Npi",
                            "GENIEReWeight_SBN_v1_multisigma_ThetaDelta2NRad",

                            # - DIS
                            "GENIEReWeight_SBN_v1_multisigma_AhtBY",
                            "GENIEReWeight_SBN_v1_multisigma_BhtBY",
                            "GENIEReWeight_SBN_v1_multisigma_CV1uBY",
                            "GENIEReWeight_SBN_v1_multisigma_CV2uBY",

                            # - COH
                            "GENIEReWeight_SBN_v1_multisigma_NormCCCOH",
                            "GENIEReWeight_SBN_v1_multisigma_NormNCCOH",

                            # - NC Elas
                            "GENIEReWeight_SBN_v1_multisigma_MaNCEL",
                            "GENIEReWeight_SBN_v1_multisigma_EtaNCEL",

                            # - FSI
                            "GENIEReWeight_SBN_v1_multisigma_MFP_pi",
                            "GENIEReWeight_SBN_v1_multisigma_FrCEx_pi",
                            "GENIEReWeight_SBN_v1_multisigma_FrInel_pi",
                            "GENIEReWeight_SBN_v1_multisigma_FrAbs_pi",
                            "GENIEReWeight_SBN_v1_multisigma_FrPiProd_pi",
                            ## -- In AR23 only
                            "GENIEReWeight_SBN_v1_multisigma_MFP_N",
                            "GENIEReWeight_SBN_v1_multisigma_FrCEx_N",
                            "GENIEReWeight_SBN_v1_multisigma_FrInel_N",
                            "GENIEReWeight_SBN_v1_multisigma_FrAbs_N",
                            "GENIEReWeight_SBN_v1_multisigma_FrPiProd_N",
                          ]

genie_multisim_knobs = [
                            # - QE
                            "GENIEReWeight_SBN_v1_multisim_CoulombCCQE",

                            # - MEC
                            "GENIEReWeight_SBN_v1_multisim_NormCCMEC",
                            "GENIEReWeight_SBN_v1_multisim_NormNCMEC",

                            # - RES
                            "GENIEReWeight_SBN_v1_multisim_RDecBR1eta",
                            "GENIEReWeight_SBN_v1_multisim_RDecBR1gamma",

                            # - Non-res pion
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvpCC1pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvpCC2pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvpNC1pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvpNC2pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvnCC2pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvnNC1pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvnNC2pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpCC1pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpCC2pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpNC1pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpNC2pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnCC1pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnCC2pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnNC1pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnNC2pi",
                        ]

In [ ]:
genie_all_knobs = genie_multisim_knobs + genie_multisigma_knobs

### make multiverse weights for multisigma genie knobs

In [ ]:
def make_multiverse_weights(nudf, evtdf, knob_list, n_univs=100):
    # 1. Create a dictionary to hold all the new columns temporarily
    new_columns_nudf = {} 
    new_columns_evtdf = {}

    for knob in knob_list:
        ## work for nudf first
        this_cols_nudf = nudf[knob].columns
        if len(this_cols_nudf) == 1: # morph
            print(f"Processing morph knob: {knob}")
            for i in range(n_univs):
                seed_input = knob + str(i) + str(0)
                np.random.seed(hash(seed_input) % (2**32))
                this_morph_weight = nudf[knob][this_cols_nudf[0]]
                wgt = 1 + (this_morph_weight - 1) * 2 * np.abs(np.random.normal(0, 1)) # std -> unc.
                wgt = np.maximum(wgt, 0) # Cap the weight to prevent negative weights
                wgt = np.minimum(wgt, 10) # Cap the weight to prevent extreme outliers
                
                # 2. Save the array to the dictionary instead of the DataFrame
                new_columns_nudf[(knob, f"univ_{i}")] = wgt
        elif len(this_cols_nudf) > 1: # multisigma
            for i in range(n_univs):
                seed_input = knob + str(i) + str(0)
                np.random.seed(hash(seed_input) % (2**32))
                this_sigma_weight = nudf[knob][this_cols_nudf[0]]
                wgt = 1 + (this_sigma_weight - 1) * np.random.normal(0, 1) # std -> unc.
                wgt = np.maximum(wgt, 0) # Cap the weight to prevent negative weights
                wgt = np.minimum(wgt, 10) # Cap the weight to prevent extreme outliers
                
                # 2. Save the array to the dictionary instead of the DataFrame
                new_columns_nudf[(knob, f"univ_{i}")] = wgt
        
        this_cols_evtdf = evtdf[knob].columns
        if len(this_cols_evtdf) == 1: # morph
            for i in range(n_univs):
                seed_input = knob + str(i) + str(1)
                np.random.seed(hash(seed_input) % (2**32))
                this_morph_weight = evtdf[knob][this_cols_evtdf[0]]
                wgt = 1 + (this_morph_weight - 1) * 2 * np.abs(np.random.normal(0, 1)) # std -> unc.
                wgt = np.maximum(wgt, 0) # Cap the weight to prevent negative weights
                wgt = np.minimum(wgt, 10) # Cap the weight to prevent extreme outliers
                
                # 2. Save the array to the dictionary instead of the DataFrame
                new_columns_evtdf[(knob, f"univ_{i}")] = wgt
        elif len(this_cols_evtdf) > 1: # multisigma
            for i in range(n_univs):
                seed_input = knob + str(i) + str(1)
                np.random.seed(hash(seed_input) % (2**32))
                this_sigma_weight = evtdf[knob][this_cols_evtdf[0]]
                wgt = 1 + (this_sigma_weight - 1) * np.random.normal(0, 1) # std -> unc.
                wgt = np.maximum(wgt, 0) # Cap the weight to prevent negative weights
                wgt = np.minimum(wgt, 10) # Cap the weight to prevent extreme outliers
                
                # 2. Save the array to the dictionary instead of the DataFrame
                new_columns_evtdf[(knob, f"univ_{i}")] = wgt
            

    # 3. Convert the dictionary to a single new DataFrame
    new_cols_nudf = pd.DataFrame(new_columns_nudf, index=nudf.index)
    new_cols_evtdf = pd.DataFrame(new_columns_evtdf, index=evtdf.index)
    
    # 4. Concatenate the original DataFrame and the new columns all at once
    nudf = pd.concat([nudf, new_cols_nudf], axis=1)
    evtdf = pd.concat([evtdf, new_cols_evtdf], axis=1)

    # 5. Synchronize univ weights for the same neutrino interactions
    for knob in knob_list:
        target_cols = [
            col for col in nudf.columns 
            if col in nudf.columns and (knob in str(col) and 'univ_' in str(col))
        ]
        
        mcnu_vals = nudf[target_cols]
        tmatch_col = next(col for col in evtdf.columns if 'tmatch_idx' in str(col))

        lookup_keys = pd.MultiIndex.from_arrays([
            evtdf.index.get_level_values('file_idx'),
            evtdf.index.get_level_values('__ntuple'),
            evtdf.index.get_level_values('entry'),
            evtdf[tmatch_col] # The target neutrino index
        ], names=evtdf.index.names)

        mapped_vals = mcnu_vals.reindex(lookup_keys)

        mapped_vals.index = evtdf.index
        evtdf[target_cols] = mapped_vals.fillna(evtdf[target_cols])
    
    
    return nudf, evtdf

In [ ]:
import pandas as pd
import numpy as np

def make_multiverse_weights(nudf, evtdf, knob_list, n_univs=100):
    # 1. Create a dictionary to hold all the new columns temporarily
    new_columns_nudf = {} 
    new_columns_evtdf = {}

    for knob in knob_list:
        ## work for nudf first
        this_cols_nudf = nudf[knob].columns
        if len(this_cols_nudf) == 1: # morph
            print(f"Processing morph knob: {knob}")
            for i in range(n_univs):
                seed_input = knob + str(i) + str(0)
                np.random.seed(hash(seed_input) % (2**32))
                this_morph_weight = nudf[knob][this_cols_nudf[0]]
                wgt = 1 + (this_morph_weight - 1) * 2 * np.abs(np.random.normal(0, 1)) # std -> unc.
                wgt = np.maximum(wgt, 0) # Cap the weight to prevent negative weights
                wgt = np.minimum(wgt, 10) # Cap the weight to prevent extreme outliers
                
                # 2. Save the array to the dictionary instead of the DataFrame
                new_columns_nudf[(knob, f"univ_{i}")] = wgt
                
        elif len(this_cols_nudf) > 1: # multisigma
            for i in range(n_univs):
                seed_input = knob + str(i) + str(0)
                np.random.seed(hash(seed_input) % (2**32))
                this_sigma_weight = nudf[knob][this_cols_nudf[0]]
                
                # Generate a single random throw to keep scales correlated within this universe
                random_throw = np.random.normal(0, 1) 
                
                if knob == "GENIEReWeight_SBN_v1_multisigma_NormCCCOH":
                    # Include 1.0 to keep the base knob, plus the requested scaled versions
                    scales = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]
                    for scale in scales:
                        wgt = 1 + scale * (this_sigma_weight - 1) * random_throw
                        wgt = np.maximum(wgt, 0)
                        wgt = np.minimum(wgt, 10)
                        
                        if scale == 1.0:
                            new_columns_nudf[(knob, f"univ_{i}")] = wgt
                        else:
                            scale_str = str(scale).replace(".", "p")
                            new_knob_name = f"{knob}_{scale_str}"
                            new_columns_nudf[(new_knob_name, f"univ_{i}")] = wgt
                else:
                    wgt = 1 + (this_sigma_weight - 1) * random_throw # std -> unc.
                    wgt = np.maximum(wgt, 0) # Cap the weight to prevent negative weights
                    wgt = np.minimum(wgt, 10) # Cap the weight to prevent extreme outliers
                    new_columns_nudf[(knob, f"univ_{i}")] = wgt
        
        ## work for evtdf second
        this_cols_evtdf = evtdf[knob].columns
        if len(this_cols_evtdf) == 1: # morph
            for i in range(n_univs):
                seed_input = knob + str(i) + str(1)
                np.random.seed(hash(seed_input) % (2**32))
                this_morph_weight = evtdf[knob][this_cols_evtdf[0]]
                wgt = 1 + (this_morph_weight - 1) * 2 * np.abs(np.random.normal(0, 1)) # std -> unc.
                wgt = np.maximum(wgt, 0) # Cap the weight to prevent negative weights
                wgt = np.minimum(wgt, 10) # Cap the weight to prevent extreme outliers
                
                # 2. Save the array to the dictionary instead of the DataFrame
                new_columns_evtdf[(knob, f"univ_{i}")] = wgt
                
        elif len(this_cols_evtdf) > 1: # multisigma
            for i in range(n_univs):
                seed_input = knob + str(i) + str(1)
                np.random.seed(hash(seed_input) % (2**32))
                this_sigma_weight = evtdf[knob][this_cols_evtdf[0]]
                
                # Generate a single random throw to keep scales correlated within this universe
                random_throw = np.random.normal(0, 1)
                
                if knob == "GENIEReWeight_SBN_v1_multisigma_NormCCCOH":
                    scales = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]
                    for scale in scales:
                        wgt = 1 + scale * (this_sigma_weight - 1) * random_throw
                        wgt = np.maximum(wgt, 0)
                        wgt = np.minimum(wgt, 10)
                        
                        if scale == 1.0:
                            new_columns_evtdf[(knob, f"univ_{i}")] = wgt
                        else:
                            scale_str = str(scale).replace(".", "p")
                            new_knob_name = f"{knob}_{scale_str}"
                            new_columns_evtdf[(new_knob_name, f"univ_{i}")] = wgt
                else:
                    wgt = 1 + (this_sigma_weight - 1) * random_throw # std -> unc.
                    wgt = np.maximum(wgt, 0) # Cap the weight to prevent negative weights
                    wgt = np.minimum(wgt, 10) # Cap the weight to prevent extreme outliers
                    new_columns_evtdf[(knob, f"univ_{i}")] = wgt
            

    # 3. Convert the dictionary to a single new DataFrame
    new_cols_nudf = pd.DataFrame(new_columns_nudf, index=nudf.index)
    new_cols_evtdf = pd.DataFrame(new_columns_evtdf, index=evtdf.index)
    
    # 4. Concatenate the original DataFrame and the new columns all at once
    nudf = pd.concat([nudf, new_cols_nudf], axis=1)
    evtdf = pd.concat([evtdf, new_cols_evtdf], axis=1)

    # 5. Synchronize univ weights for the same neutrino interactions
    for knob in knob_list:
        # Note: Because the scaled knob names (e.g., GENIEReWeight_..._0p9) 
        # still contain the original `knob` string, this target_cols logic 
        # naturally picks them up and synchronizes them without needing changes!
        target_cols = [
            col for col in nudf.columns 
            if col in nudf.columns and (knob in str(col) and 'univ_' in str(col))
        ]
        
        mcnu_vals = nudf[target_cols]
        tmatch_col = next(col for col in evtdf.columns if 'tmatch_idx' in str(col))

        lookup_keys = pd.MultiIndex.from_arrays([
            evtdf.index.get_level_values('file_idx'),
            evtdf.index.get_level_values('__ntuple'),
            evtdf.index.get_level_values('entry'),
            evtdf[tmatch_col] # The target neutrino index
        ], names=evtdf.index.names)

        mapped_vals = mcnu_vals.reindex(lookup_keys)

        mapped_vals.index = evtdf.index
        evtdf[target_cols] = mapped_vals.fillna(evtdf[target_cols])
    
    return nudf, evtdf

In [ ]:
mc_bnb_cosmic_dfs['mcnu'], mc_bnb_cosmic_dfs['evt'] = make_multiverse_weights(mc_bnb_cosmic_dfs['mcnu'], mc_bnb_cosmic_dfs['evt'], genie_multisigma_knobs)

### make meged flux and g4 syst columns

In [ ]:
def cap_weights_by_syst(df_dict, syst_name, threshold=10.0):
    """
    Caps weights for a specific systematic group within a multi-indexed dataframe.
    
    Parameters:
    df_dict (DataFrame): The dataframe slice containing the events (e.g., mc_bnb_cosmic_dfs['evt'])
    syst_name (str): The name of the systematic uncertainty to cap.
    threshold (float): The maximum allowed weight value.
    
    Returns:
    DataFrame: The updated dataframe.
    """
    # Check if the systematic name exists in the top level of the MultiIndex
    if syst_name in df_dict.columns.levels[0]:
        # .clip(upper=threshold) caps all 1000 universes at once
        df_dict[syst_name] = df_dict[syst_name].clip(upper=threshold)
        print(f"Success: Capped weights for '{syst_name}' at {threshold}.")
    else:
        print(f"Warning: Systematic '{syst_name}' not found in the dataframe.")
    
    return df_dict

cap G4 weights to 10.0

In [ ]:
mc_bnb_cosmic_dfs['mcnu'] = cap_weights_by_syst(mc_bnb_cosmic_dfs['mcnu'], "reinteractions_neutron_Geant4", threshold=10.0)
mc_bnb_cosmic_dfs['evt'] = cap_weights_by_syst(mc_bnb_cosmic_dfs['evt'], "reinteractions_neutron_Geant4", threshold=10.0)

In [ ]:
def merge_syst_weights(df, base_knobs, merged_name,nuniv=1000):
    new_cols_data = {}
    
    for iuniv in range(nuniv):
        univ_label = f'univ_{iuniv}'
        
        # 1. Construct the exact tuple column names we need for this universe
        # e.g., [('expskin_Flux', 'univ_0'), ('horncurrent_Flux', 'univ_0'), ...]
        target_cols = [(knob, univ_label) for knob in base_knobs]
        
        # 2. Safety check: Ensure these columns actually exist in the DataFrame
        # This prevents KeyErrors if a universe is missing a specific knob
        valid_cols = [col for col in target_cols if col in df.columns]
        
        if not valid_cols:
            continue
            
        # 3. Extract raw numpy array for these specific columns and multiply across rows
        # axis=1 multiplies horizontally across the columns
        new_cols_data[(merged_name, univ_label)] = np.prod(df[valid_cols].values, axis=1)

    # 4. Build the final DataFrame from the dictionary
    if not new_cols_data:
        print("Warning: No matching columns found. Returning original DataFrame.")
        return df
        
    new_df = pd.DataFrame(new_cols_data, index=df.index)
    
    # 5. Apply the MultiIndex structure to match your original DataFrame perfectly
    if isinstance(df.columns, pd.MultiIndex):
        new_df.columns = pd.MultiIndex.from_tuples(
            new_df.columns, 
            names=df.columns.names
        )

    # 6. Concatenate once at the end for maximum memory efficiency
    return pd.concat([df, new_df], axis=1)

In [ ]:
mc_bnb_cosmic_dfs['mcnu'] = merge_syst_weights(mc_bnb_cosmic_dfs['mcnu'], flux_multisim_knobs, "Flux", nuniv=1000)
mc_bnb_cosmic_dfs['mcnu'] = merge_syst_weights(mc_bnb_cosmic_dfs['mcnu'], G4_multisim_knobs, "G4", nuniv=1000)

mc_bnb_cosmic_dfs['evt'] = merge_syst_weights(mc_bnb_cosmic_dfs['evt'], flux_multisim_knobs, "Flux", nuniv=1000)
mc_bnb_cosmic_dfs['evt'] = merge_syst_weights(mc_bnb_cosmic_dfs['evt'], G4_multisim_knobs, "G4", nuniv=1000)

mc_bnb_cosmic_dfs['mcnu'] = merge_syst_weights(mc_bnb_cosmic_dfs['mcnu'], genie_multisim_knobs + genie_multisigma_knobs, "GENIE", nuniv=1000)
mc_bnb_cosmic_dfs['evt'] = merge_syst_weights(mc_bnb_cosmic_dfs['evt'], genie_multisim_knobs + genie_multisigma_knobs, "GENIE", nuniv=1000)

In [ ]:
## process genie reweight with NormCCCOH_0p5
genie_all_knobs_normcccoh_05 = [
    "GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p5" if knob == "GENIEReWeight_SBN_v1_multisigma_NormCCCOH" else knob 
    for knob in genie_all_knobs
]
genie_all_knobs_normcccoh_05
mc_bnb_cosmic_dfs['mcnu'] = merge_syst_weights(mc_bnb_cosmic_dfs['mcnu'], genie_all_knobs_normcccoh_05, "GENIE_NormCCCOH_0p5", nuniv=1000)
mc_bnb_cosmic_dfs['evt'] = merge_syst_weights(mc_bnb_cosmic_dfs['evt'], genie_all_knobs_normcccoh_05, "GENIE_NormCCCOH_0p5", nuniv=1000)

### add mc stat weights

In [ ]:
from numpy.random import Generator, PCG64, SeedSequence

def get_MCstat_unc_ultra_fast(nu_df, n_universes=100, seed=42):
    n_events = len(nu_df)
    
    # 1. Create a SeedSequence based on a global entropy (e.g., 42)
    # This is the "Master Seed" for your entire experiment
    ss = SeedSequence(seed)
    
    # 2. Spawn independent sequences for each universe
    # This ensures Universe 1 is statistically independent from Universe 2
    child_seeds = ss.spawn(n_universes)
    
    # Pre-allocate the matrix (using int16 or int32 to save RAM)
    # 2.4M rows * 100 cols * 4 bytes approx 960MB
    poisson_weights = np.zeros((n_events, n_universes), dtype=np.int32)

    print(f"Vectorizing {n_universes} universes for {n_events} rows...")
    
    for uidx in range(n_universes):
        # Create a high-performance Generator for this specific universe
        # We use the spawned seed to ensure reproducibility
        rg = Generator(PCG64(child_seeds[uidx]))
        
        # KEY STEP: Generate the entire column at once in C-code (No inner loop!)
        # This is ~1000x faster than looping through rows
        poisson_weights[:, uidx] = rg.poisson(lam=1.0, size=n_events)

    # 3. Create MultiIndex Columns
    cols = pd.MultiIndex.from_product(
        [["MCstat"], [f"univ_{i}" for i in range(n_universes)]]
    )

    # 4. Construct DataFrame and Join
    # Joining 2.4M rows can be slow; we do it efficiently
    mcstat_df = pd.DataFrame(poisson_weights, index=nu_df.index, columns=cols)
    
    print("Joining results to main DataFrame...")
    nu_df = pd.concat([nu_df, mcstat_df], axis=1)
    
    return nu_df, poisson_weights.T

In [ ]:
mc_bnb_cosmic_dfs['evt'], _ = get_MCstat_unc_ultra_fast(mc_bnb_cosmic_dfs['evt'], n_universes=100, seed=42)
mc_bnb_cosmic_dfs['mcnu'], _ = get_MCstat_unc_ultra_fast(mc_bnb_cosmic_dfs['mcnu'], n_universes=100, seed=43)

In [ ]:
## synchronize MCstat multiverses for evt_df with mcnu_df for the same neutrino interactions

# 1. Identify the target columns (safely handling the tuple column names)
target_cols = [
    col for col in mc_bnb_cosmic_dfs['evt'].columns 
    if col in mc_bnb_cosmic_dfs['mcnu'].columns and ('MCstat' in str(col) and 'univ_' in str(col))
]

# 2. Isolate just those columns from the true neutrino dataframe
mcnu_vals = mc_bnb_cosmic_dfs['mcnu'][target_cols]

# 4. Find the exact column key for 'tmatch_idx' 
tmatch_col = next(col for col in mc_bnb_cosmic_dfs['evt'].columns if 'tmatch_idx' in str(col))

# 5. Construct a lookup index using evt_df's data
lookup_keys = pd.MultiIndex.from_arrays([
    mc_bnb_cosmic_dfs['evt'].index.get_level_values('file_idx'),
    mc_bnb_cosmic_dfs['evt'].index.get_level_values('__ntuple'),
    mc_bnb_cosmic_dfs['evt'].index.get_level_values('entry'),
    mc_bnb_cosmic_dfs['evt'][tmatch_col] # The target neutrino index
], names=mc_bnb_cosmic_dfs['mcnu'].index.names)

# 6. Reindex mcnu_vals using our lookup keys (now safe because mcnu_vals is perfectly unique)
mapped_vals = mcnu_vals.reindex(lookup_keys)

# 7. Force mapped_vals to share evt_df's exact index 
mapped_vals.index = mc_bnb_cosmic_dfs['evt'].index

# 8. Overwrite the columns in evt_df in-place
mc_bnb_cosmic_dfs['evt'][target_cols] = mapped_vals.fillna(mc_bnb_cosmic_dfs['evt'][target_cols])

In [ ]:
mc_bnb_cosmic_dfs['evt'].MCstat

## Make sr and sb dfs

In [ ]:
mc_bnb_cosmic_dfs['evt_sr'] = mc_bnb_cosmic_dfs['evt'][mc_bnb_cosmic_dfs['evt'].reco_t < 0.04]
mc_bnb_cosmic_dfs['evt_sb'] = mc_bnb_cosmic_dfs['evt'][mc_bnb_cosmic_dfs['evt'].reco_t > 0.06]

data_bnb_light_dfs['evt_sr'] = data_bnb_light_dfs['evt'][data_bnb_light_dfs['evt'].reco_t < 0.04]
data_bnb_light_dfs['evt_sb'] = data_bnb_light_dfs['evt'][data_bnb_light_dfs['evt'].reco_t > 0.06]

data_offbeam_light_dfs['evt_sr'] = data_offbeam_light_dfs['evt'][data_offbeam_light_dfs['evt'].reco_t < 0.04]
data_offbeam_light_dfs['evt_sb'] = data_offbeam_light_dfs['evt'][data_offbeam_light_dfs['evt'].reco_t > 0.06]

## Bining Optimizations

### Functions for optimization

- Optimize with adapitve binning for a given number of bins: ensure similar stats
- Then look for maximum number of bins statisfies purity requirement (fraction of events staying in diagonal components in smeaering matrix)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random

def plot_matrices(raw_hist, bin_edges, purity, efficiency, var_label="Variable", plot_title="Detector Model", out_title_prefix="muon_momentum"):
    """
    Plots the Smear Matrix and Response Matrix side-by-side.
    Saves the output to a PDF before displaying.
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # 1. Smear Matrix (Normalize by True Bin Selected events -> Shape/Migration)
    # axis=0 corresponds to the columns (True bins)
    col_sums = raw_hist.sum(axis=0, keepdims=True)
    smear_matrix = np.divide(raw_hist, col_sums, 
                             out=np.zeros_like(raw_hist, dtype=float), 
                             where=col_sums!=0)
    
    # 2. Response Matrix (Multiply by Efficiency -> Survival + Smearing)
    # Reshape efficiency to broadcast across the columns
    response_matrix = smear_matrix * efficiency.reshape(1, -1)
    
    tick_positions = np.arange(len(bin_edges))
    labels = np.round(bin_edges, 3)

    # --- PLOT 1: Smear Matrix (Left) ---
    sns.heatmap(smear_matrix, annot=True, fmt=".2f", cmap="YlGnBu", ax=axes[0],
                xticklabels=False, yticklabels=False)
    axes[0].invert_yaxis()
    axes[0].set_xticks(tick_positions)
    axes[0].set_xticklabels(labels, rotation=45, ha='right')
    axes[0].set_yticks(tick_positions)
    axes[0].set_yticklabels(labels, rotation=0)
    axes[0].set_title(f"{plot_title}: Smear Matrix\n(Migration Only: Col Sum = 1.0)")
    axes[0].set_xlabel(f"True {var_label} Bin Edges")          
    axes[0].set_ylabel(f"Reconstructed {var_label} Bin Edges") 

    # --- PLOT 2: Response Matrix (Right) ---
    sns.heatmap(response_matrix, annot=True, fmt=".3f", cmap="OrRd", ax=axes[1],
                xticklabels=False, yticklabels=False)
    axes[1].invert_yaxis()
    axes[1].set_xticks(tick_positions)
    axes[1].set_xticklabels(labels, rotation=45, ha='right')
    axes[1].set_yticks(tick_positions)
    axes[1].set_yticklabels(labels, rotation=0)
    axes[1].set_title(f"{plot_title}: Response Matrix\n(Efficiency + Smearing: Col Sum = $\\epsilon$)")
    axes[1].set_xlabel(f"True {var_label} Bin Edges")          
    axes[1].set_ylabel(f"Reconstructed {var_label} Bin Edges") 
    
    plt.tight_layout()
    
    # Save first, then show
    plt.savefig(f"{out_title_prefix}_smear_n_response_opt.pdf", format='pdf', bbox_inches='tight')
    plt.show()
    
    # Print metrics
    print("\n" + "=" * 50)
    print("FINAL BINNING SUMMARY")
    print("=" * 50)
    print(f"Number of Bins: {len(bin_edges) - 1}")
    print(f"Bin Edges:      {np.round(bin_edges, 3)}")
    print(f"Purity:         {np.round(purity, 3)}")
    print(f"Efficiency:     {np.round(efficiency, 3)}")
    print("=" * 50 + "\n")

In [ ]:
def optimize_binning_monte_carlo(evt_df, evt_df_col, nu_df, nu_df_col, 
                                 iterations=10000, step_size=0.05,
                                 min_events=15, min_bin_completeness=0.50, 
                                 trace_shift_threshold=0.68,
                                 max_bins=15, min_bins=4,
                                 bin_edge_min=None, bin_edge_max=None, 
                                 random_seed=42, # <-- ADDED SEED PARAMETER
                                 var_label="Variable", plot_title="Detector Model",
                                 plot_title_prefix="output"):
    """
    Uses Monte Carlo to find the best configuration. 
    Tracks the overall best, AND the best configuration for each specific n_bins.
    Includes a random seed for strict reproducibility.
    """
    # ==========================================
    # Set the random seeds for reproducibility
    # ==========================================
    if random_seed is not None:
        random.seed(random_seed)
        np.random.seed(random_seed)
        
    # 1. Filter for signal
    evt_signal = evt_df[evt_df['nuint_categ'] == 1]
    nu_signal = nu_df[nu_df['nuint_categ'] == 1]
    
    reco_vals = evt_signal[evt_df_col].values
    true_vals_selected = evt_signal[nu_df_col].values
    true_vals_all = nu_signal[nu_df_col].values
    
    val_min = bin_edge_min if bin_edge_min is not None else np.floor(np.min(reco_vals))
    val_max = bin_edge_max if bin_edge_max is not None else np.ceil(np.max(reco_vals))
    
    grid = np.arange(val_min + step_size, val_max, step_size)
    
    print(f"Starting Monte Carlo Global Scan for {var_label}...")
    print(f"Evaluating {iterations} configurations (Seed: {random_seed})...")
    
    # Tracking the OVERALL champion
    best_global_trace = -1
    best_min_comp = -1
    best_edges = None
    best_comp_per_bin = None
    best_hist = None
    
    # Tracking the champion FOR EACH n_bins
    best_configs_per_nbins = {} 
    valid_configs_found = 0

    # 2. The Monte Carlo Loop
    for i in range(iterations):
        n_bins = random.randint(min_bins, max_bins)
        internal_edges = sorted(random.sample(list(grid), n_bins - 1))
        current_edges = np.array([val_min] + internal_edges + [val_max])
        
        raw_hist, _, _ = np.histogram2d(reco_vals, true_vals_selected, bins=current_edges)
        reco_counts = np.sum(raw_hist, axis=1) # Row sums
        true_counts = np.sum(raw_hist, axis=0) # Column sums (Selected True)
        
        # Fast fail: stats check
        if np.any(reco_counts < min_events):
            continue
            
        # Calculate COMPLETENESS (Normalize by Column / True Counts)
        comp_per_bin = np.divide(np.diag(raw_hist), true_counts, 
                                 out=np.zeros_like(true_counts, dtype=float), 
                                 where=true_counts!=0)
        
        # Fast fail: hard floor check for Completeness
        if np.any(comp_per_bin < min_bin_completeness):
            continue
            
        valid_configs_found += 1
        
        total_events = np.sum(raw_hist)
        global_trace = np.sum(np.diag(raw_hist)) / total_events if total_events > 0 else 0
        current_min_bin_comp = np.min(comp_per_bin)
        
        # ==========================================
        # Two-Tiered Ranking Logic (Reusable function)
        # ==========================================
        def is_new_champ(new_trace, new_min, old_trace, old_min, threshold):
            if old_trace < 0: return True # First valid one found
            if new_trace >= threshold and old_trace >= threshold:
                if new_min > old_min: return True
                if new_min == old_min and new_trace > old_trace: return True
            elif new_trace >= threshold and old_trace < threshold:
                return True
            elif new_trace < threshold and old_trace < threshold:
                if new_trace > old_trace: return True
            return False

        # Check if it's the best for this SPECIFIC number of bins
        old_n_trace = best_configs_per_nbins[n_bins]['global_trace'] if n_bins in best_configs_per_nbins else -1
        old_n_min = best_configs_per_nbins[n_bins]['min_comp'] if n_bins in best_configs_per_nbins else -1
        
        if is_new_champ(global_trace, current_min_bin_comp, old_n_trace, old_n_min, trace_shift_threshold):
            best_configs_per_nbins[n_bins] = {
                'edges': current_edges,
                'global_trace': global_trace,
                'min_comp': current_min_bin_comp,
                'comp_per_bin': comp_per_bin,
                'hist': raw_hist
            }

        # Check if it's the OVERALL global best
        if is_new_champ(global_trace, current_min_bin_comp, best_global_trace, best_min_comp, trace_shift_threshold):
            best_global_trace = global_trace
            best_min_comp = current_min_bin_comp
            best_edges = current_edges
            best_comp_per_bin = comp_per_bin
            best_hist = raw_hist

    # 3. Post-Loop Processing
    if best_edges is None:
        print("\n--> FAILED: Could not find any configuration that satisfied the constraints.")
        return None, None, None, None, None
        
    print(f"\n--> Finished! Evaluated {iterations} layouts. Found {valid_configs_found} valid configurations.")
    
    # Print the Summary Table for each n_bins
    print("\n" + "=" * 60)
    print(f"{'Bins':<6} | {'Global Trace':<15} | {'Min Bin Comp':<15} | {'Phase Ended'}")
    print("-" * 60)
    for n in sorted(best_configs_per_nbins.keys()):
        cfg = best_configs_per_nbins[n]
        phase = "Phase 2 (Min Comp)" if cfg['global_trace'] >= trace_shift_threshold else "Phase 1 (Trace)"
        marker = " <-- OVERALL CHAMPION" if np.array_equal(cfg['edges'], best_edges) else ""
        print(f"{n:<6} | {cfg['global_trace']:<15.3f} | {cfg['min_comp']:<15.3f} | {phase}{marker}")
    print("=" * 60 + "\n")
    
    true_counts_total, _ = np.histogram(true_vals_all, bins=best_edges)
    reco_counts_from_true = np.sum(best_hist, axis=0) 
    best_efficiency = np.divide(reco_counts_from_true, true_counts_total,
                                out=np.zeros_like(true_counts_total, dtype=float), 
                                where=true_counts_total!=0)
    
    # Call your plotter
    plot_matrices(best_hist, best_edges, best_comp_per_bin, best_efficiency, var_label, plot_title, plot_title_prefix)
    
    # Returning the dictionary as a 5th element so you can inspect other bin counts!
    return best_edges, best_comp_per_bin, best_efficiency, best_hist, best_configs_per_nbins

In [ ]:
def optimize_binning_monte_carlo(evt_df, evt_df_col, nu_df, nu_df_col, 
                                 iterations=10000, step_size=0.05,
                                 min_events=15, min_bin_completeness=0.50, 
                                 trace_shift_threshold=0.68,
                                 max_bins=15, min_bins=4,
                                 bin_edge_min=None, bin_edge_max=None, 
                                 random_seed=42, 
                                 var_label="Variable", plot_title="Detector Model",
                                 plot_title_prefix="output"):
    """
    Uses Monte Carlo to find the best configuration. 
    Tracks the overall best, AND the best configuration for each specific n_bins.
    Includes a random seed for strict reproducibility.
    """
    # ==========================================
    # Set the random seeds for reproducibility
    # ==========================================
    if random_seed is not None:
        random.seed(random_seed)
        np.random.seed(random_seed)
        
    # 1. Filter for signal
    evt_signal = evt_df[evt_df['nuint_categ'] == 1]
    nu_signal = nu_df[nu_df['nuint_categ'] == 1]
    
    reco_vals = evt_signal[evt_df_col].values
    true_vals_selected = evt_signal[nu_df_col].values
    true_vals_all = nu_signal[nu_df_col].values
    
    val_min = bin_edge_min if bin_edge_min is not None else np.floor(np.min(reco_vals))
    val_max = bin_edge_max if bin_edge_max is not None else np.ceil(np.max(reco_vals))
    
    grid = np.arange(val_min + step_size, val_max, step_size)
    
    print(f"Starting Monte Carlo Global Scan for {var_label}...")
    print(f"Evaluating {iterations} configurations (Seed: {random_seed})...")
    
    # Tracking the OVERALL champion
    best_global_trace = -1
    best_min_comp = -1
    best_edges = None
    best_comp_per_bin = None
    best_hist = None
    
    # Tracking the champion FOR EACH n_bins
    best_configs_per_nbins = {} 
    valid_configs_found = 0

    # 2. The Monte Carlo Loop
    for i in range(iterations):
        n_bins = random.randint(min_bins, max_bins)
        internal_edges = sorted(random.sample(list(grid), n_bins - 1))
        current_edges = np.array([val_min] + internal_edges + [val_max])
        
        raw_hist, _, _ = np.histogram2d(reco_vals, true_vals_selected, bins=current_edges)
        reco_counts = np.sum(raw_hist, axis=1) # Row sums
        true_counts = np.sum(raw_hist, axis=0) # Column sums (Selected True)
        
        # Fast fail: stats check
        if np.any(reco_counts < min_events):
            continue
            
        # Calculate COMPLETENESS (Normalize by Column / True Counts)
        comp_per_bin = np.divide(np.diag(raw_hist), true_counts, 
                                 out=np.zeros_like(true_counts, dtype=float), 
                                 where=true_counts!=0)
        
        # Fast fail: hard floor check for Completeness
        if np.any(comp_per_bin < min_bin_completeness):
            continue
            
        valid_configs_found += 1
        
        total_events = np.sum(raw_hist)
        global_trace = np.sum(np.diag(raw_hist)) / total_events if total_events > 0 else 0
        current_min_bin_comp = np.min(comp_per_bin)
        
        # ==========================================
        # Two-Tiered Ranking Logic (Reusable function)
        # ==========================================
        def is_new_champ(new_trace, new_min, old_trace, old_min, threshold):
            if old_trace < 0: return True # First valid one found
            if new_trace >= threshold and old_trace >= threshold:
                if new_min > old_min: return True
                if new_min == old_min and new_trace > old_trace: return True
            elif new_trace >= threshold and old_trace < threshold:
                return True
            elif new_trace < threshold and old_trace < threshold:
                if new_trace > old_trace: return True
            return False

        # Check if it's the best for this SPECIFIC number of bins
        old_n_trace = best_configs_per_nbins[n_bins]['global_trace'] if n_bins in best_configs_per_nbins else -1
        old_n_min = best_configs_per_nbins[n_bins]['min_comp'] if n_bins in best_configs_per_nbins else -1
        
        if is_new_champ(global_trace, current_min_bin_comp, old_n_trace, old_n_min, trace_shift_threshold):
            best_configs_per_nbins[n_bins] = {
                'edges': current_edges,
                'global_trace': global_trace,
                'min_comp': current_min_bin_comp,
                'comp_per_bin': comp_per_bin,
                'hist': raw_hist
            }

        # Check if it's the OVERALL global best
        if is_new_champ(global_trace, current_min_bin_comp, best_global_trace, best_min_comp, trace_shift_threshold):
            best_global_trace = global_trace
            best_min_comp = current_min_bin_comp
            best_edges = current_edges
            best_comp_per_bin = comp_per_bin
            best_hist = raw_hist

    # 3. Post-Loop Processing
    if best_edges is None:
        print("\n--> FAILED: Could not find any configuration that satisfied the constraints.")
        return None, None, None, None, None
        
    print(f"\n--> Finished! Evaluated {iterations} layouts. Found {valid_configs_found} valid configurations.")
    
    # Print the Summary Table for each n_bins
    print("\n" + "=" * 60)
    print(f"{'Bins':<6} | {'Global Trace':<15} | {'Min Bin Comp':<15} | {'Phase Ended'}")
    print("-" * 60)
    for n in sorted(best_configs_per_nbins.keys()):
        cfg = best_configs_per_nbins[n]
        phase = "Phase 2 (Min Comp)" if cfg['global_trace'] >= trace_shift_threshold else "Phase 1 (Trace)"
        marker = " <-- OVERALL CHAMPION" if np.array_equal(cfg['edges'], best_edges) else ""
        print(f"{n:<6} | {cfg['global_trace']:<15.3f} | {cfg['min_comp']:<15.3f} | {phase}{marker}")
    print("=" * 60 + "\n")
    
    # =========================================================
    # NEW: Show events in each bin for the best overall layout
    # =========================================================
    best_reco_counts = np.sum(best_hist, axis=1) # Sum across rows to get total reco events per bin
    print("=" * 60)
    print(f"OVERALL CHAMPION: RECO EVENTS PER BIN ({len(best_edges)-1} Bins)")
    print("-" * 60)
    print(f"{'Bin Index':<10} | {'Bin Edges':<25} | {'Events (Reco)':<15}")
    print("-" * 60)
    for i in range(len(best_reco_counts)):
        edge_range = f"[{best_edges[i]:.2f}, {best_edges[i+1]:.2f})"
        print(f"{i:<10} | {edge_range:<25} | {best_reco_counts[i]:<15.0f}")
    print("=" * 60 + "\n")
    # =========================================================
    
    true_counts_total, _ = np.histogram(true_vals_all, bins=best_edges)
    reco_counts_from_true = np.sum(best_hist, axis=0) 
    best_efficiency = np.divide(reco_counts_from_true, true_counts_total,
                                out=np.zeros_like(true_counts_total, dtype=float), 
                                where=true_counts_total!=0)
    
    # Call your plotter
    plot_matrices(best_hist, best_edges, best_comp_per_bin, best_efficiency, var_label, plot_title, plot_title_prefix)
    
    # Returning the dictionary as a 5th element so you can inspect other bin counts!
    return best_edges, best_comp_per_bin, best_efficiency, best_hist, best_configs_per_nbins

In [ ]:
range_p_mu_bin_edges, _, _, _, _ = optimize_binning_monte_carlo(mc_bnb_cosmic_dfs['evt_sr'], ('range_p_mu', ''), mc_bnb_cosmic_dfs['mcnu'], ('true_p_mu', ''), iterations = 100000, min_events=25., bin_edge_min=0., bin_edge_max=3., min_bins=3, step_size = 0.01, var_label=r"$P_\mu$ (GeV/c)", plot_title="Muon Momentum", plot_title_prefix="muon_momentum")

In [ ]:
cos_theta_mu_bin_edges, _, _, _, _ = optimize_binning_monte_carlo(mc_bnb_cosmic_dfs['evt_sr'], ('long_dirz', ''), mc_bnb_cosmic_dfs['mcnu'], ('true_cos_theta_mu', ''), iterations = 100000, min_events=25., bin_edge_min=-1., bin_edge_max=1., max_bins = 5, min_bins = 3, step_size = 0.001, var_label=r"$\cos\theta_\mu$", plot_title="Muon Cosine Theta", plot_title_prefix="muon_cos_theta")

In [ ]:
cos_theta_pi_bin_edges, _, _, _, _ = optimize_binning_monte_carlo(mc_bnb_cosmic_dfs['evt_sr'], ('short_dirz', ''), mc_bnb_cosmic_dfs['mcnu'], ('true_cos_theta_pi', ''), iterations = 100000, min_events=25., bin_edge_min=-1., bin_edge_max=1., max_bins = 5, min_bins = 4, step_size = 0.001, var_label=r"$\cos\theta_{\pi}$", plot_title="Pion Cosine Theta", plot_title_prefix="pion_cos_theta")

### With containment requirement

- Binning optimizatio results with 0.4 tager purity
- P_mu: [0.    0.399 0.505 0.657 0.764 0.906 6.   ]
- cos_mu: [-1.     0.929  0.951  0.974  0.986  0.994  0.997  1.   ]
- cos_pi: [-1.     0.815  0.863  0.898  0.937  0.964  0.977  0.987  1.   ]

- Muon containment requirement provides much better response
- But for charged pion, no gain in response.

In [ ]:
mc_bnb_cosmic_dfs['evt_sr_contained'] = mc_bnb_cosmic_dfs['evt_sr'][(mc_bnb_cosmic_dfs['evt_sr'].is_muon_contained) & (mc_bnb_cosmic_dfs['evt_sr'].is_cpi_contained)]

In [ ]:
range_p_mu_bin_edges_contained, _, _, _, _ = optimize_binning_monte_carlo(mc_bnb_cosmic_dfs['evt_sr_contained'], ('range_p_mu', ''), mc_bnb_cosmic_dfs['mcnu'], ('true_p_mu', ''),  iterations = 100000, min_events=25., max_bins = 10, min_bins=4, bin_edge_min=0., bin_edge_max=6., var_label=r"$P_\mu$ (GeV/c)", plot_title="Muon Momentum", plot_title_prefix="muon_momentum_contained")

In [ ]:
range_p_pi_bin_edges_contained, _, _, _, _ = optimize_binning_monte_carlo(mc_bnb_cosmic_dfs['evt_sr_contained'], ('range_p_cpi', ''), mc_bnb_cosmic_dfs['mcnu'], ('true_p_pi', ''),  iterations = 100000, min_events=25., max_bins = 10, min_bins=3, bin_edge_min=0., bin_edge_max=1., var_label=r"$P_\pi$ (GeV/c)", plot_title="Pion Momentum", plot_title_prefix="pion_momentum_contained")

### Check stopping charged pions

- Stopping charged pions selected with chi2_mu < 6 cut provides reasonable momentum response

In [ ]:
draw_a_distribution(mc_bnb_cosmic_dfs['evt_sr_contained'], ('short_trk_chi2_mu', ''), title_x=r"$\chi^2_\mu$", x_min = 0., x_max = 20., nbins=40, title_y="Events", is_logx = False, is_logy = False, label_top = "", label_SBND = "SBND Internal", out_name = None)

In [ ]:
mc_bnb_cosmic_dfs['evt_sr_contained_stoping_cpi'] = mc_bnb_cosmic_dfs['evt_sr_contained'][mc_bnb_cosmic_dfs['evt_sr_contained'].short_trk_chi2_mu < 5.]

In [ ]:
range_p_pi_bin_edges_stopping_, _, _, _, _ = optimize_binning_monte_carlo(mc_bnb_cosmic_dfs['evt_sr_contained_stoping_cpi'], ('range_p_cpi', ''), mc_bnb_cosmic_dfs['mcnu'], ('true_p_pi', ''), iterations = 100000, min_events=25., min_bins=2, bin_edge_min=0., bin_edge_max=6., var_label=r"$P_\pi$ (GeV/c)", plot_title="Pion Momentum", plot_title_prefix="pion_momentum_contained_stopping_cpi")

## Define Varconfig with the binnings

In [ ]:
varcfg_p_mu = VariableConfig(
    var_save_name="muon-p",
    var_plot_name="P_\mu",
    var_labels=[r"$\mathrm{P_\mu}$ (GeV/c)", 
    r"$\mathrm{P_\mu^{reco.}}$ (GeV/c)", 
    r"$\mathrm{P_\mu^{true}}$ (GeV/c)"],
    bins=range_p_mu_bin_edges,
    var_evt_reco_col='range_p_mu',
    var_evt_truth_col='true_p_mu',
    var_nu_col='true_p_mu',
    xsec_label=r"$\frac{d\sigma}{dP_\mu}$ ($\mathrm{cm}^2$ / GeV/c)"
)

varcfg_cos_mu = VariableConfig(
    var_save_name="muon-cos",
    var_plot_name="\cos\theta_{\mu^\mp}",
    var_labels=[r"$\cos\theta_{\mu^\mp}$",
    r"$\cos\theta_{\mu^\mp}^{reco.}$", 
    r"$\cos\theta_{\mu^\mp}^{true}$"],
    bins=cos_theta_mu_bin_edges,
    var_evt_reco_col='long_dirz',
    var_evt_truth_col='true_cos_theta_mu',
    var_nu_col='true_cos_theta_mu',
    xsec_label=r"$\frac{d\sigma}{d\cos\theta_{\mu^\mp}}$ ($\mathrm{cm}^2$)"
)

varcfg_cos_pi = VariableConfig(
    var_save_name="pion-cos",
    var_plot_name="\cos\theta_{\pi^\pm}",
    var_labels=[r"$\cos\theta_{\pi^\pm}$", 
    r"$\cos\theta_{\pi^\pm}^{reco.}$", 
    r"$\cos\theta_{\pi^\pm}^{true}$"],
    bins=cos_theta_pi_bin_edges,
    var_evt_reco_col='short_dirz',
    var_evt_truth_col='true_cos_theta_pi',
    var_nu_col='true_cos_theta_pi',
    xsec_label=r"$\frac{d\sigma}{d\cos\theta_{\pi^\pm}}$ ($\mathrm{cm}^2$)"
)

In [ ]:
p_mu_objs = {}
p_mu_objs['varcfg'] = varcfg_p_mu

cos_mu_objs = {}
cos_mu_objs['varcfg'] = varcfg_cos_mu

cos_pi_objs = {}
cos_pi_objs['varcfg'] = varcfg_cos_pi

## Normalization to Cross-section Unit

In [ ]:
fluxfile = "/data/sungbino/sbnd/flux/sbnd_original_flux.root"
flux = uproot.open(fluxfile)
numu_flux = flux["flux_sbnd_numu"].to_numpy()
anumu_flux = flux["flux_sbnd_anumu"].to_numpy()
numu_flux_vals = numu_flux[0]
anumu_flux_vals = anumu_flux[0]

INTEGRATED_FLUX = (numu_flux_vals.sum() + anumu_flux_vals.sum()) / (1e4  * 1e6) # to cm2 # to POT
print("Integrated flux (numu + anumu) over total MC POT: %.3e per (cm^2 POT)" % INTEGRATED_FLUX)

In [ ]:
TPC_Vol = (180. * 2.) * (190. * 2) * (250. - 10.) + (180. * 2.) * (190. + 100.) * (450. - 250.) # cm^3

## Signal Histograms

In [ ]:
p_mu_objs["signal_res"] = signal_hists(evtdf=mc_bnb_cosmic_dfs['evt_sr'], nudf=mc_bnb_cosmic_dfs['mcnu'], var_config=varcfg_p_mu, save_fig=False, save_name=None, return_data=True)
cos_mu_objs["signal_res"] = signal_hists(evtdf=mc_bnb_cosmic_dfs['evt_sr'], nudf=mc_bnb_cosmic_dfs['mcnu'], var_config=varcfg_cos_mu, save_fig=False, save_name=None, return_data=True)
cos_pi_objs["signal_res"] = signal_hists(evtdf=mc_bnb_cosmic_dfs['evt_sr'], nudf=mc_bnb_cosmic_dfs['mcnu'], var_config=varcfg_cos_pi, save_fig=False, save_name=None, return_data=True)


## BNB + light data and Off-beam + light counts

In [ ]:
p_mu_objs['sr_bnb_light_events'], _ = np.histogram(get_clipped_evts(data_bnb_light_dfs['evt_sr'], p_mu_objs["varcfg"].var_evt_reco_col, p_mu_objs["varcfg"].bins)[0], p_mu_objs["varcfg"].bins)
p_mu_objs['sb_bnb_light_events'], _ = np.histogram(get_clipped_evts(data_bnb_light_dfs['evt_sb'], p_mu_objs["varcfg"].var_evt_reco_col, p_mu_objs["varcfg"].bins)[0], p_mu_objs["varcfg"].bins)

cos_mu_objs['sr_bnb_light_events'], _ = np.histogram(get_clipped_evts(data_bnb_light_dfs['evt_sr'], cos_mu_objs["varcfg"].var_evt_reco_col, cos_mu_objs["varcfg"].bins)[0], cos_mu_objs["varcfg"].bins)
cos_mu_objs['sb_bnb_light_events'], _ = np.histogram(get_clipped_evts(data_bnb_light_dfs['evt_sb'], cos_mu_objs["varcfg"].var_evt_reco_col, cos_mu_objs["varcfg"].bins)[0], cos_mu_objs["varcfg"].bins)

cos_pi_objs['sr_bnb_light_events'], _ = np.histogram(get_clipped_evts(data_bnb_light_dfs['evt_sr'], cos_pi_objs["varcfg"].var_evt_reco_col, cos_pi_objs["varcfg"].bins)[0], cos_pi_objs["varcfg"].bins)
cos_pi_objs['sb_bnb_light_events'], _ = np.histogram(get_clipped_evts(data_bnb_light_dfs['evt_sb'], cos_pi_objs["varcfg"].var_evt_reco_col, cos_pi_objs["varcfg"].bins)[0], cos_pi_objs["varcfg"].bins)

In [ ]:
p_mu_objs['sr_offbeam_light_events'], _ = np.histogram(get_clipped_evts(data_offbeam_light_dfs['evt_sr'], p_mu_objs["varcfg"].var_evt_reco_col, p_mu_objs["varcfg"].bins)[0], p_mu_objs["varcfg"].bins)
p_mu_objs['sb_offbeam_light_events'], _ = np.histogram(get_clipped_evts(data_offbeam_light_dfs['evt_sb'], p_mu_objs["varcfg"].var_evt_reco_col, p_mu_objs["varcfg"].bins)[0], p_mu_objs["varcfg"].bins)

cos_mu_objs['sr_offbeam_light_events'], _ = np.histogram(get_clipped_evts(data_offbeam_light_dfs['evt_sr'], cos_mu_objs["varcfg"].var_evt_reco_col, cos_mu_objs["varcfg"].bins)[0], cos_mu_objs["varcfg"].bins)
cos_mu_objs['sb_offbeam_light_events'], _ = np.histogram(get_clipped_evts(data_offbeam_light_dfs['evt_sb'], cos_mu_objs["varcfg"].var_evt_reco_col, cos_mu_objs["varcfg"].bins)[0], cos_mu_objs["varcfg"].bins)

cos_pi_objs['sr_offbeam_light_events'], _ = np.histogram(get_clipped_evts(data_offbeam_light_dfs['evt_sr'], cos_pi_objs["varcfg"].var_evt_reco_col, cos_pi_objs["varcfg"].bins)[0], cos_pi_objs["varcfg"].bins)
cos_pi_objs['sb_offbeam_light_events'], _ = np.histogram(get_clipped_evts(data_offbeam_light_dfs['evt_sb'], cos_pi_objs["varcfg"].var_evt_reco_col, cos_pi_objs["varcfg"].bins)[0], cos_pi_objs["varcfg"].bins)

In [ ]:
p_mu_objs['sr_offbeam_light_events'] = p_mu_objs['sr_offbeam_light_events'] * intime_gate_scale
p_mu_objs['sb_offbeam_light_events'] = p_mu_objs['sb_offbeam_light_events'] * intime_gate_scale

cos_mu_objs['sr_offbeam_light_events'] = cos_mu_objs['sr_offbeam_light_events'] * intime_gate_scale
cos_mu_objs['sb_offbeam_light_events'] = cos_mu_objs['sb_offbeam_light_events'] * intime_gate_scale

cos_pi_objs['sr_offbeam_light_events'] = cos_pi_objs['sr_offbeam_light_events'] * intime_gate_scale
cos_pi_objs['sb_offbeam_light_events'] = cos_pi_objs['sb_offbeam_light_events'] * intime_gate_scale

## Event Rates in Multiverse and CV 

In [ ]:
def process_universes(obj_list, mc_dfs, mc_params, cov_type="rate", sys_label="Flux"):
    """
    Processes flux universes and scales results by POT for a list of analysis objects.
    
    Args:
        obj_list: List of dictionaries (e.g., [p_mu_objs, cos_mu_objs, cos_pi_objs])
        mc_dfs: Dictionary containing 'evt_sr', 'evt_sb', and 'mcnu'
        mc_params: Dictionary containing 'flux', 'pot', 'vol', 'topology_list', and 'scale'
    """
    regions = ["sr", "sb"]
    categories = ["signal", "bkg"]
    
    for obj in obj_list:
        for reg in regions:
            # 1. Fetch Universe Data
            (obj[f"{reg}_signal_univ_events_" + sys_label + "_" + cov_type], 
             obj[f"{reg}_signal_sel_reco_cv_" + cov_type], 
             obj[f"{reg}_bkg_univ_events_" + sys_label + "_" + cov_type], 
             obj[f"{reg}_bkg_sel_reco_cv_" + cov_type]) = get_genie_univs(
                cov_type, 
                mc_dfs[f'evt_{reg}'], 
                mc_dfs['mcnu'], 
                obj['varcfg'], 
                (sys_label,), 
                flux=mc_params['flux'], 
                pot=mc_params['pot'], 
                vol=mc_params['vol'], 
                topology_list=mc_params['topology_list']
            )
            
            # 2. Apply POT Scaling
            # This scales all 4 keys just generated for this specific region
            keys_to_scale = [
                f"{reg}_signal_univ_events_" + sys_label + "_" + cov_type, 
                f"{reg}_signal_sel_reco_cv_" + cov_type,
                f"{reg}_bkg_univ_events_" + sys_label + "_" + cov_type, 
                f"{reg}_bkg_sel_reco_cv_" + cov_type
            ]
            
            for key in keys_to_scale:
                obj[key] = obj[key] * mc_params['scale']

# --- Execution ---

# Pack your parameters into a config dict for cleanliness
mc_config = {
    'flux': INTEGRATED_FLUX,
    'pot': mc_tot_pot,
    'vol': TPC_Vol,
    'topology_list': mode_list,
    'scale': mc_pot_scale
}

In [ ]:
from multiprocessing import Pool
from functools import partial
from tqdm import tqdm

# 1. The Top-Level Worker (Must be outside the main function for pickling)
def _process_single_knob_worker(sys_label, obj_list, mc_dfs, mc_params, cov_type):
    """
    Worker function: Processes ONE knob and returns the computed data dictionaries.
    """
    regions = ["sr", "sb"]
    knob_results = []
    
    for obj in obj_list:
        obj_result = {}
        for reg in regions:
            # Fetch Universe Data
            sig_univ, sig_cv, bkg_univ, bkg_cv = get_genie_univs(
                cov_type, 
                mc_dfs[f'evt_{reg}'], 
                mc_dfs['mcnu'], 
                obj['varcfg'], 
                (sys_label,), 
                flux=mc_params['flux'], 
                pot=mc_params['pot'], 
                vol=mc_params['vol'], 
                topology_list=mc_params['topology_list']
            )
            
            # Apply POT Scaling
            scale = mc_params['scale']
            obj_result[f"{reg}_signal_univ_events_{sys_label}_{cov_type}"] = sig_univ * scale
            obj_result[f"{reg}_signal_sel_reco_cv_{cov_type}"] = sig_cv * scale
            obj_result[f"{reg}_bkg_univ_events_{sys_label}_{cov_type}"] = bkg_univ * scale
            obj_result[f"{reg}_bkg_sel_reco_cv_{cov_type}"] = bkg_cv * scale
            
        knob_results.append(obj_result)
        
    return sys_label, knob_results


# 2. The Main Wrapper Function
def process_all_universes_parallel(obj_list, knob_list, mc_dfs, mc_params, cov_type="rate", n_cores=6):
    """
    Orchestrates the parallel processing of multisim knobs and merges results.
    """
    # Freeze the constant arguments for the worker
    worker_func = partial(
        _process_single_knob_worker, 
        obj_list=obj_list, 
        mc_dfs=mc_dfs, 
        mc_params=mc_params,
        cov_type=cov_type
    )

    print(f"Spawning {n_cores} workers for {len(knob_list)} knobs...")

    # Execute parallel pool
    with Pool(processes=n_cores) as pool:
        results = list(tqdm(
            pool.imap(worker_func, knob_list), 
            total=len(knob_list),
            desc="Processing Universes"
        ))

    print("Merging parallel results...")
    
    # Merge the returned dictionaries back into the main obj_list
    for sys_label, knob_results in results:
        for original_obj, new_data in zip(obj_list, knob_results):
            original_obj.update(new_data)

    print("Done!")
    return obj_list

In [ ]:
def plot_univ_hists_all_signal(knob, cov_type="rate"):
    plot_univ_hists(p_mu_objs["sr_signal_univ_events_" + knob + "_" + cov_type], p_mu_objs["sr_signal_sel_reco_cv_" + cov_type], knob, p_mu_objs["varcfg"], categ_name="Signal Region: Signal", save_fig=True, save_name="./multiverses/p_mu_sr_signal_" + knob.lower() + "_univ_events.pdf")
    plot_univ_hists(p_mu_objs["sb_signal_univ_events_" + knob + "_" + cov_type], p_mu_objs["sb_signal_sel_reco_cv_" + cov_type], knob, p_mu_objs["varcfg"], categ_name="Sideband: Signal", save_fig=True, save_name="./multiverses/p_mu_sb_signal_" + knob.lower() + "_univ_events.pdf")

    plot_univ_hists(cos_mu_objs["sr_signal_univ_events_" + knob + "_" + cov_type], cos_mu_objs["sr_signal_sel_reco_cv_" + cov_type], knob, cos_mu_objs["varcfg"], categ_name="Signal Region: Signal", save_fig=True, save_name="./multiverses/cos_mu_sr_signal_" + knob.lower() + "_univ_events.pdf")
    plot_univ_hists(cos_mu_objs["sb_signal_univ_events_" + knob + "_" + cov_type], cos_mu_objs["sb_signal_sel_reco_cv_" + cov_type], knob, cos_mu_objs["varcfg"], categ_name="Sideband: Signal", save_fig=True, save_name="./multiverses/cos_mu_sb_signal_" + knob.lower() + "_univ_events.pdf")

    plot_univ_hists(cos_pi_objs["sr_signal_univ_events_" + knob + "_" + cov_type], cos_pi_objs["sr_signal_sel_reco_cv_" + cov_type], knob, cos_pi_objs["varcfg"], categ_name="Signal Region: Signal", save_fig=True, save_name="./multiverses/cos_pi_sr_signal_" + knob.lower() + "_univ_events.pdf")
    plot_univ_hists(cos_pi_objs["sb_signal_univ_events_" + knob + "_" + cov_type], cos_pi_objs["sb_signal_sel_reco_cv_" + cov_type], knob, cos_pi_objs["varcfg"], categ_name="Sideband: Signal", save_fig=True, save_name="./multiverses/cos_pi_sb_signal_" + knob.lower() + "_univ_events.pdf")

def plot_univ_hists_all_signal_xsec(knob, cov_type="xsec"):
    plot_univ_hists(p_mu_objs["sr_signal_univ_events_" + knob + "_" + cov_type], p_mu_objs["sr_signal_sel_reco_cv_" + cov_type], knob, p_mu_objs["varcfg"], categ_name="Signal Region: Signal", save_fig=True, save_name="./multiverses/p_mu_sr_signal_" + knob.lower() + "_univ_events.pdf", ylabel=p_mu_objs["varcfg"].xsec_label)
    plot_univ_hists(p_mu_objs["sb_signal_univ_events_" + knob + "_" + cov_type], p_mu_objs["sb_signal_sel_reco_cv_" + cov_type], knob, p_mu_objs["varcfg"], categ_name="Sideband: Signal", save_fig=True, save_name="./multiverses/p_mu_sb_signal_" + knob.lower() + "_univ_events.pdf", ylabel=p_mu_objs["varcfg"].xsec_label)

    plot_univ_hists(cos_mu_objs["sr_signal_univ_events_" + knob + "_" + cov_type], cos_mu_objs["sr_signal_sel_reco_cv_" + cov_type], knob, cos_mu_objs["varcfg"], categ_name="Signal Region: Signal", save_fig=True, save_name="./multiverses/cos_mu_sr_signal_" + knob.lower() + "_univ_events.pdf", ylabel=cos_mu_objs["varcfg"].xsec_label)
    plot_univ_hists(cos_mu_objs["sb_signal_univ_events_" + knob + "_" + cov_type], cos_mu_objs["sb_signal_sel_reco_cv_" + cov_type], knob, cos_mu_objs["varcfg"], categ_name="Sideband: Signal", save_fig=True, save_name="./multiverses/cos_mu_sb_signal_" + knob.lower() + "_univ_events.pdf", ylabel=cos_mu_objs["varcfg"].xsec_label)

    plot_univ_hists(cos_pi_objs["sr_signal_univ_events_" + knob + "_" + cov_type], cos_pi_objs["sr_signal_sel_reco_cv_" + cov_type], knob, cos_pi_objs["varcfg"], categ_name="Signal Region: Signal", save_fig=True, save_name="./multiverses/cos_pi_sr_signal_" + knob.lower() + "_univ_events.pdf", ylabel=cos_pi_objs["varcfg"].xsec_label)
    plot_univ_hists(cos_pi_objs["sb_signal_univ_events_" + knob + "_" + cov_type], cos_pi_objs["sb_signal_sel_reco_cv_" + cov_type], knob, cos_pi_objs["varcfg"], categ_name="Sideband: Signal", save_fig=True, save_name="./multiverses/cos_pi_sb_signal_" + knob.lower() + "_univ_events.pdf", ylabel=cos_pi_objs["varcfg"].xsec_label)

def plot_univ_hists_all_bkg(knob, cov_type="rate"):
    plot_univ_hists(p_mu_objs["sr_bkg_univ_events_" + knob + "_" + cov_type], p_mu_objs["sr_bkg_sel_reco_cv_" + cov_type], knob, p_mu_objs["varcfg"], categ_name="Signal Region: Background", save_fig=True, save_name="./multiverses/p_mu_sr_bkg_" + knob.lower() + "_univ_events.pdf")
    plot_univ_hists(p_mu_objs["sb_bkg_univ_events_" + knob + "_" + cov_type], p_mu_objs["sb_bkg_sel_reco_cv_" + cov_type], knob, p_mu_objs["varcfg"], categ_name="Sideband: Background", save_fig=True, save_name="./multiverses/p_mu_sb_bkg_" + knob.lower() + "_univ_events.pdf")

    plot_univ_hists(cos_mu_objs["sr_bkg_univ_events_" + knob + "_" + cov_type], cos_mu_objs["sr_bkg_sel_reco_cv_" + cov_type], knob, cos_mu_objs["varcfg"], categ_name="Signal Region: Background", save_fig=True, save_name="./multiverses/cos_mu_sr_bkg_" + knob.lower() + "_univ_events.pdf")
    plot_univ_hists(cos_mu_objs["sb_bkg_univ_events_" + knob + "_" + cov_type], cos_mu_objs["sb_bkg_sel_reco_cv_" + cov_type], knob, cos_mu_objs["varcfg"], categ_name="Sideband: Background", save_fig=True, save_name="./multiverses/cos_mu_sb_bkg_" + knob.lower() + "_univ_events.pdf")

    plot_univ_hists(cos_pi_objs["sr_bkg_univ_events_" + knob + "_" + cov_type], cos_pi_objs["sr_bkg_sel_reco_cv_" + cov_type], knob, cos_pi_objs["varcfg"], categ_name="Signal Region: Background", save_fig=True, save_name="./multiverses/cos_pi_sr_bkg_" + knob.lower() + "_univ_events.pdf")
    plot_univ_hists(cos_pi_objs["sb_bkg_univ_events_" + knob + "_" + cov_type], cos_pi_objs["sb_bkg_sel_reco_cv_" + cov_type], knob, cos_pi_objs["varcfg"], categ_name="Sideband: Background", save_fig=True, save_name="./multiverses/cos_pi_sb_bkg_" + knob.lower() + "_univ_events.pdf")

In [ ]:
_ = process_all_universes_parallel([p_mu_objs, cos_mu_objs, cos_pi_objs], genie_multisigma_knobs, mc_bnb_cosmic_dfs, mc_config, cov_type="rate", n_cores=10)
_ = process_all_universes_parallel([p_mu_objs, cos_mu_objs, cos_pi_objs], genie_multisim_knobs, mc_bnb_cosmic_dfs, mc_config, cov_type="rate", n_cores=10)
_ = process_all_universes_parallel([p_mu_objs, cos_mu_objs, cos_pi_objs], flux_multisim_knobs, mc_bnb_cosmic_dfs, mc_config, cov_type="rate", n_cores=10)
_ = process_all_universes_parallel([p_mu_objs, cos_mu_objs, cos_pi_objs], G4_multisim_knobs, mc_bnb_cosmic_dfs, mc_config, cov_type="rate", n_cores=10)

## Study GENIE Knobs

### Total MC Bkg

In [ ]:
process_universes([p_mu_objs, cos_mu_objs, cos_pi_objs], mc_bnb_cosmic_dfs, mc_config, cov_type="rate", sys_label="GENIE")

In [ ]:
process_universes([p_mu_objs, cos_mu_objs, cos_pi_objs], mc_bnb_cosmic_dfs, mc_config, cov_type="rate", sys_label="GENIE_NormCCCOH_0p5")

In [ ]:
p_mu_objs["sr_total_bkg_cv_rate"] = p_mu_objs["sr_bkg_sel_reco_cv_rate"].sum(axis=0)
p_mu_objs["sr_total_bkg_univ_events_GENIE_rate"] = p_mu_objs["sr_bkg_univ_events_GENIE_rate"].sum(axis=1)
plot_univ_hists(p_mu_objs["sr_total_bkg_univ_events_GENIE_rate"], p_mu_objs["sr_total_bkg_cv_rate"], "GENIE", p_mu_objs['varcfg'], categ_name="Total MC Background in Signal Region",  save_fig=True, save_name="./multiverses/p_mu_sr_bkg_genie_univ_events.pdf")

cos_mu_objs["sr_total_bkg_cv_rate"] = cos_mu_objs["sr_bkg_sel_reco_cv_rate"].sum(axis=0)
cos_mu_objs["sr_total_bkg_univ_events_GENIE_rate"] = cos_mu_objs["sr_bkg_univ_events_GENIE_rate"].sum(axis=1)
plot_univ_hists(cos_mu_objs["sr_total_bkg_univ_events_GENIE_rate"], cos_mu_objs["sr_total_bkg_cv_rate"], "GENIE", cos_mu_objs['varcfg'], categ_name="Total MC Background in Signal Region",  save_fig=True, save_name="./multiverses/cos_mu_sr_bkg_genie_univ_events.pdf")

cos_pi_objs["sr_total_bkg_cv_rate"] = cos_pi_objs["sr_bkg_sel_reco_cv_rate"].sum(axis=0)
cos_pi_objs["sr_total_bkg_univ_events_GENIE_rate"] = cos_pi_objs["sr_bkg_univ_events_GENIE_rate"].sum(axis=1)
plot_univ_hists(cos_pi_objs["sr_total_bkg_univ_events_GENIE_rate"], cos_pi_objs["sr_total_bkg_cv_rate"], "GENIE", cos_pi_objs['varcfg'], categ_name="Total MC Background in Signal Region",  save_fig=True, save_name="./multiverses/cos_pi_sr_bkg_genie_univ_events.pdf")

In [ ]:
p_mu_objs["sb_total_bkg_cv_rate"] = p_mu_objs["sb_bkg_sel_reco_cv_rate"].sum(axis=0)
p_mu_objs["sb_total_bkg_univ_events_GENIE_rate"] = p_mu_objs["sb_bkg_univ_events_GENIE_rate"].sum(axis=1)
plot_univ_hists(p_mu_objs["sb_total_bkg_univ_events_GENIE_rate"], p_mu_objs["sb_total_bkg_cv_rate"], "GENIE", p_mu_objs['varcfg'], categ_name="Total MC Background in Sideband",  save_fig=True, save_name="./multiverses/p_mu_sb_bkg_genie_univ_events.pdf")

cos_mu_objs["sb_total_bkg_cv_rate"] = cos_mu_objs["sb_bkg_sel_reco_cv_rate"].sum(axis=0)
cos_mu_objs["sb_total_bkg_univ_events_GENIE_rate"] = cos_mu_objs["sb_bkg_univ_events_GENIE_rate"].sum(axis=1)
plot_univ_hists(cos_mu_objs["sb_total_bkg_univ_events_GENIE_rate"], cos_mu_objs["sb_total_bkg_cv_rate"], "GENIE", cos_mu_objs['varcfg'], categ_name="Total MC Background in Sideband",  save_fig=True, save_name="./multiverses/cos_mu_sb_bkg_genie_univ_events.pdf")

cos_pi_objs["sb_total_bkg_cv_rate"] = cos_pi_objs["sb_bkg_sel_reco_cv_rate"].sum(axis=0)
cos_pi_objs["sb_total_bkg_univ_events_GENIE_rate"] = cos_pi_objs["sb_bkg_univ_events_GENIE_rate"].sum(axis=1)
plot_univ_hists(cos_pi_objs["sb_total_bkg_univ_events_GENIE_rate"], cos_pi_objs["sb_total_bkg_cv_rate"], "GENIE", cos_pi_objs['varcfg'], categ_name="Total MC Background in Sideband",  save_fig=True, save_name="./multiverses/cos_pi_sb_bkg_genie_univ_events.pdf")

### Study for Signal Unc

Multiverse shapes with the default weights

In [ ]:
plot_univ_hists_all_signal("GENIEReWeight_SBN_v1_multisigma_NormCCCOH", cov_type="rate")

In [ ]:
process_universes([p_mu_objs, cos_mu_objs, cos_pi_objs], mc_bnb_cosmic_dfs, mc_config, cov_type="xsec", sys_label="GENIEReWeight_SBN_v1_multisigma_NormCCCOH")

In [ ]:
plot_univ_hists_all_signal_xsec("GENIEReWeight_SBN_v1_multisigma_NormCCCOH", cov_type="xsec")

In [ ]:
process_universes([p_mu_objs, cos_mu_objs, cos_pi_objs], mc_bnb_cosmic_dfs, mc_config, cov_type="rate", sys_label="GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p9")
process_universes([p_mu_objs, cos_mu_objs, cos_pi_objs], mc_bnb_cosmic_dfs, mc_config, cov_type="rate", sys_label="GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p8")
process_universes([p_mu_objs, cos_mu_objs, cos_pi_objs], mc_bnb_cosmic_dfs, mc_config, cov_type="rate", sys_label="GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p7")
process_universes([p_mu_objs, cos_mu_objs, cos_pi_objs], mc_bnb_cosmic_dfs, mc_config, cov_type="rate", sys_label="GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p6")
process_universes([p_mu_objs, cos_mu_objs, cos_pi_objs], mc_bnb_cosmic_dfs, mc_config, cov_type="rate", sys_label="GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p5")

In [ ]:
plot_univ_hists_all_signal("GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p9", cov_type="rate")
plot_univ_hists_all_signal("GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p8", cov_type="rate")
plot_univ_hists_all_signal("GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p7", cov_type="rate")
plot_univ_hists_all_signal("GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p6", cov_type="rate")
plot_univ_hists_all_signal("GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p5", cov_type="rate")

In [ ]:
process_universes([p_mu_objs, cos_mu_objs, cos_pi_objs], mc_bnb_cosmic_dfs, mc_config, cov_type="xsec", sys_label="GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p9")
process_universes([p_mu_objs, cos_mu_objs, cos_pi_objs], mc_bnb_cosmic_dfs, mc_config, cov_type="xsec", sys_label="GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p8")
process_universes([p_mu_objs, cos_mu_objs, cos_pi_objs], mc_bnb_cosmic_dfs, mc_config, cov_type="xsec", sys_label="GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p7")
process_universes([p_mu_objs, cos_mu_objs, cos_pi_objs], mc_bnb_cosmic_dfs, mc_config, cov_type="xsec", sys_label="GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p6")
process_universes([p_mu_objs, cos_mu_objs, cos_pi_objs], mc_bnb_cosmic_dfs, mc_config, cov_type="xsec", sys_label="GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p5")

In [ ]:
plot_univ_hists_all_signal_xsec("GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p9", cov_type="xsec")
plot_univ_hists_all_signal_xsec("GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p8", cov_type="xsec")
plot_univ_hists_all_signal_xsec("GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p7", cov_type="xsec")
plot_univ_hists_all_signal_xsec("GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p6", cov_type="xsec")
plot_univ_hists_all_signal_xsec("GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p5", cov_type="xsec")

In [ ]:
print(mc_bnb_cosmic_dfs['evt_sr'][mc_bnb_cosmic_dfs['evt_sr'].nuint_categ==1].GENIEReWeight_SBN_v1_multisigma_NormCCCOH)

## Multiverses for the merged flux and g4 knobs

In [ ]:
# Run the function on your three objects
process_universes([p_mu_objs, cos_mu_objs, cos_pi_objs], mc_bnb_cosmic_dfs, mc_config, cov_type="rate", sys_label="Flux")
process_universes([p_mu_objs, cos_mu_objs, cos_pi_objs], mc_bnb_cosmic_dfs, mc_config, cov_type="rate", sys_label="G4")
process_universes([p_mu_objs, cos_mu_objs, cos_pi_objs], mc_bnb_cosmic_dfs, mc_config, cov_type="rate", sys_label="MCstat")



In [ ]:
p_mu_objs.keys()

## Save objs

In [ ]:
import joblib

data_to_save = {
    "p_mu_objs": p_mu_objs,
    "cos_mu_objs": cos_mu_objs,
    "cos_pi_objs": cos_pi_objs,
    "data_tot_pot": data_tot_pot,
}

# 2. Save the dictionary to a single file
joblib.dump(data_to_save, 'multisim_objects.joblib')
print("Successfully saved all objects!")

### Multiverses Signal

In [ ]:
plot_univ_hists(p_mu_objs["sr_signal_univ_events_Flux_rate"], p_mu_objs["sr_signal_sel_reco_cv_rate"], "Flux", p_mu_objs["varcfg"], categ_name="Signal Region: Signal", save_fig=True, save_name="./multiverses/p_mu_sr_signal_flux_univ_events.pdf")
plot_univ_hists(p_mu_objs["sb_signal_univ_events_Flux_rate"], p_mu_objs["sb_signal_sel_reco_cv_rate"], "Flux", p_mu_objs["varcfg"], categ_name="Sideband: Signal", save_fig=True, save_name="./multiverses/p_mu_sb_signal_flux_univ_events.pdf")

plot_univ_hists(cos_mu_objs["sr_signal_univ_events_Flux_rate"], cos_mu_objs["sr_signal_sel_reco_cv_rate"], "Flux", cos_mu_objs["varcfg"], categ_name="Signal Region: Signal", save_fig=True, save_name="./multiverses/cos_mu_sr_signal_flux_univ_events.pdf")
plot_univ_hists(cos_mu_objs["sb_signal_univ_events_Flux_rate"], cos_mu_objs["sb_signal_sel_reco_cv_rate"], "Flux", cos_mu_objs["varcfg"], categ_name="Sideband: Signal", save_fig=True, save_name="./multiverses/cos_mu_sb_signal_flux_univ_events.pdf")

plot_univ_hists(cos_pi_objs["sr_signal_univ_events_Flux_rate"], cos_pi_objs["sr_signal_sel_reco_cv_rate"], "Flux", cos_pi_objs["varcfg"], categ_name="Signal Region: Signal", save_fig=True, save_name="./multiverses/cos_pi_sr_signal_flux_univ_events.pdf")
plot_univ_hists(cos_pi_objs["sb_signal_univ_events_Flux_rate"], cos_pi_objs["sb_signal_sel_reco_cv_rate"], "Flux", cos_pi_objs["varcfg"], categ_name="Sideband: Signal", save_fig=True, save_name="./multiverses/cos_pi_sb_signal_flux_univ_events.pdf")

### Multiverses: bkg

#### total bkg

In [ ]:
def plot_univ_hists_all_bkg(knob="Flux"):
    for reg in ["sr", "sb"]:
        p_mu_objs[reg + "_total_bkg_cv_rate"] = p_mu_objs[reg + "_bkg_sel_reco_cv_rate"].sum(axis=0)
        p_mu_objs[reg + "_total_bkg_univ_events_" + knob + "_rate"] = p_mu_objs[reg + "_bkg_univ_events_" + knob + "_rate"].sum(axis=1)
        plot_univ_hists(p_mu_objs[reg + "_total_bkg_univ_events_" + knob + "_rate"], p_mu_objs[reg + "_total_bkg_cv_rate"], knob, p_mu_objs['varcfg'], categ_name="Total MC Background")
    
        cos_mu_objs[reg + "_total_bkg_cv_rate"] = cos_mu_objs[reg + "_bkg_sel_reco_cv_rate"].sum(axis=0)
        cos_mu_objs[reg + "_total_bkg_univ_events_" + knob + "_rate"] = cos_mu_objs[reg + "_bkg_univ_events_" + knob + "_rate"].sum(axis=1)
        plot_univ_hists(cos_mu_objs[reg + "_total_bkg_univ_events_" + knob + "_rate"], cos_mu_objs[reg + "_total_bkg_cv_rate"], knob, cos_mu_objs['varcfg'], categ_name="Total MC Background")

        cos_pi_objs[reg + "_total_bkg_cv_rate"] = cos_pi_objs[reg + "_bkg_sel_reco_cv_rate"].sum(axis=0)
        cos_pi_objs[reg + "_total_bkg_univ_events_" + knob + "_rate"] = cos_pi_objs[reg + "_bkg_univ_events_" + knob + "_rate"].sum(axis=1)
        plot_univ_hists(cos_pi_objs[reg + "_total_bkg_univ_events_" + knob + "_rate"], cos_pi_objs[reg + "_total_bkg_cv_rate"], knob, cos_pi_objs['varcfg'], categ_name="Total MC Background")

In [ ]:
plot_univ_hists_all_bkg("Flux")
plot_univ_hists_all_bkg("G4")
plot_univ_hists_all_bkg("MCstat")

### Total MC Event Rate: Signal + Bkg

In [ ]:
def plot_univ_hists_all_mc(knob="Flux"):
    for reg in ["sr", "sb"]:
        p_mu_objs[reg + "_total_mc_cv_rate"] = p_mu_objs[reg + "_total_bkg_cv_rate"] + p_mu_objs[reg + "_signal_sel_reco_cv_rate"]
        p_mu_objs[reg + "_total_mc_univ_events_" + knob + "_rate"] = p_mu_objs[reg + "_total_bkg_univ_events_" + knob + "_rate"] + p_mu_objs[reg + "_signal_univ_events_" + knob + "_rate"]
        plot_univ_hists(p_mu_objs[reg + '_total_mc_univ_events_' + knob + '_rate'], p_mu_objs[reg + '_total_mc_cv_rate'], knob, p_mu_objs['varcfg'], categ_name="Signal Region: Total MC Prediction", save_fig=True, save_name="./multiverses/p_mu_" + reg + "_total_mc_" + knob.lower() + "_univ_events.pdf")



In [ ]:
plot_univ_hists_all_mc("Flux")
plot_univ_hists_all_mc("G4")
plot_univ_hists_all_mc("MCstat")